# 03 Regression First Model (New Notebook)

## What changes first (before coding)

This notebook changes the ML task from binary late/not-late to **regression on delivery margin days**.

- Old task: predict `is_late` directly.
- New task: predict `delivery_margin_days = actual_delivery_days - estimated_delivery_days`.

Why this is better:
- It models the size of delay/earliness, not just a binary flag.
- It keeps more signal for ML than residual binary late labels.
- It still supports operations decisions.

## ML roles in this new setup

1. **Prediction role:** estimate expected margin days for each order.
2. **Risk role:** convert predicted margin to probability/risk bands and late flags.
3. **Decision role:** apply capacity-constrained thresholds (Top-K or cap) for intervention.

This keeps strong ML depth while staying business-ready.

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.isotonic import IsotonicRegression

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

sns.set_theme(style="whitegrid")

DATA_PATH = Path("../datasets/processed/olist_merged_clean.csv")
df = pd.read_csv(DATA_PATH)

for c in [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

print("Loaded shape:", df.shape)
print("Late rate (%):", round(df["is_late"].mean() * 100, 2))
print("XGBoost available:", HAS_XGB)

Loaded shape: (96189, 34)
Late rate (%): 7.94
XGBoost available: True


In [7]:
work = df.copy()

# Core regression targets
work["actual_delivery_days"] = (
    work["order_delivered_customer_date"] - work["order_purchase_timestamp"]
).dt.days

work["estimated_delivery_days"] = (
    work["order_estimated_delivery_date"] - work["order_purchase_timestamp"]
).dt.days

# Positive => delivered before estimate, Negative => delivered after estimate
work["delivery_margin_days"] = (
    work["order_estimated_delivery_date"] - work["order_delivered_customer_date"]
).dt.days

# Safety cleaning on target logic
work = work.replace([np.inf, -np.inf], np.nan)
work = work.dropna(subset=["order_purchase_timestamp", "actual_delivery_days", "delivery_margin_days"])
work = work[(work["actual_delivery_days"] >= 0) & (work["actual_delivery_days"] <= 120)]
work = work[(work["delivery_margin_days"] >= -120) & (work["delivery_margin_days"] <= 120)]

# Time-based split boundaries
train_end = work["order_purchase_timestamp"].quantile(0.70)
val_end = work["order_purchase_timestamp"].quantile(0.85)

split = np.where(
    work["order_purchase_timestamp"] <= train_end,
    "train",
    np.where(work["order_purchase_timestamp"] <= val_end, "val", "test"),
)
work["split"] = split

print("Prepared shape:", work.shape)
print(work[["actual_delivery_days", "delivery_margin_days"]].describe().round(2))
print("Split counts:\n", work["split"].value_counts())

Prepared shape: (96142, 36)
       actual_delivery_days  delivery_margin_days
count              96142.00              96142.00
mean                  12.01                 11.00
std                    8.98                  9.58
min                    0.00               -106.00
25%                    6.00                  6.00
50%                   10.00                 12.00
75%                   15.00                 16.00
max                  118.00                108.00
Split counts:
 split
train    67299
test     14422
val      14421
Name: count, dtype: int64


In [14]:
TARGET = "delivery_margin_days"

leakage_cols = {
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "actual_delivery_days",
    "delivery_margin_days",
    "delivery_delay_days",
    "is_late",
    "order_id",
}

# Exclude fields that are available only after the order outcome.
post_outcome_tokens = ("delivered", "review", "status")

candidate_cols = [
    c
    for c in work.columns
    if c not in leakage_cols
    and c not in {"split"}
    and not any(t in c.lower() for t in post_outcome_tokens)
]

X_train = work.loc[work["split"] == "train", candidate_cols].copy()
y_train = work.loc[work["split"] == "train", TARGET].copy()

X_val = work.loc[work["split"] == "val", candidate_cols].copy()
y_val = work.loc[work["split"] == "val", TARGET].copy()

X_test = work.loc[work["split"] == "test", candidate_cols].copy()
y_test = work.loc[work["split"] == "test", TARGET].copy()

for frame in (X_train, X_val, X_test):
    for c in frame.columns:
        if pd.api.types.is_datetime64_any_dtype(frame[c]):
            frame[c] = frame[c].astype("int64", copy=False) // 10**9

num_cols = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
cat_cols = [c for c in X_train.columns if c not in num_cols]

numeric_pipe = Pipeline(
    steps=[
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler(with_mean=False)),
    ]
)

categorical_pipe = Pipeline(
    steps=[
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01)),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop",
)

print(f"Feature columns: {len(candidate_cols)} | Numeric: {len(num_cols)} | Categorical: {len(cat_cols)}")

Feature columns: 28 | Numeric: 22 | Categorical: 6


In [15]:
models = {
    "elastic_net": ElasticNet(alpha=0.0008, l1_ratio=0.2, random_state=42, max_iter=4000),
    "rf_reg": RandomForestRegressor(
        n_estimators=300,
        max_depth=18,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=42,
    ),
}

if HAS_XGB:
    models["xgb_reg"] = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=7,
        subsample=0.9,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    )

val_rows = []
fitted = {}

for name, model in models.items():
    pipe = Pipeline(steps=[("prep", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)

    p_train = pipe.predict(X_train)
    p_val = pipe.predict(X_val)

    row = {
        "model": name,
        "train_mae": mean_absolute_error(y_train, p_train),
        "val_mae": mean_absolute_error(y_val, p_val),
        "val_rmse": np.sqrt(mean_squared_error(y_val, p_val)),
        "val_r2": r2_score(y_val, p_val),
    }
    val_rows.append(row)
    fitted[name] = pipe

val_table = pd.DataFrame(val_rows).sort_values("val_mae")
val_table.round(4)

C:\Users\Devyansh\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_coordinate_descent.py:658: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 76005.89023778867, tolerance: 627.8608478075453
  model = cd_fast.sparse_enet_coordinate_descent(


,model,train_mae,val_mae,val_rmse,val_r2
2,xgb_reg,3.8084,5.0666,7.3167,0.4423
1,rf_reg,3.1905,5.8780,7.8629,0.3559
0,elastic_net,5.3086,6.9945,8.2816,0.2855


In [16]:
best_name = val_table.iloc[0]["model"]
best_pipe = fitted[best_name]

pred_val = best_pipe.predict(X_val)
pred_test = best_pipe.predict(X_test)

test_metrics = {
    "best_model": best_name,
    "test_mae": mean_absolute_error(y_test, pred_test),
    "test_rmse": np.sqrt(mean_squared_error(y_test, pred_test)),
    "test_r2": r2_score(y_test, pred_test),
}

pd.DataFrame([test_metrics]).round(4)

,best_model,test_mae,test_rmse,test_r2
0,xgb_reg,5.1173,7.3109,0.2304


In [17]:
# Convert regression output into a late-risk score: P(margin <= 0)
# We calibrate from validation predictions to keep probability semantics.
val_is_late = (y_val <= 0).astype(int)

def build_risk_mapper(pred_margin_val, y_late_val):
    order = np.argsort(pred_margin_val)
    x = pred_margin_val[order]
    y = y_late_val.values[order]

    # Monotonic: lower predicted margin => higher late risk
    iso = IsotonicRegression(y_min=0.0, y_max=1.0, increasing=False, out_of_bounds="clip")
    iso.fit(x, y)
    return iso

risk_mapper = build_risk_mapper(pred_val, val_is_late)
risk_test = risk_mapper.predict(pred_test)

policy_df = pd.DataFrame({
    "pred_margin": pred_test,
    "risk_late": risk_test,
    "is_late": (y_test <= 0).astype(int).values,
})

# Capacity-constrained thresholding (max alerts = 20% of orders)
max_alert_rate = 0.20
thr_grid = np.linspace(0.05, 0.95, 91)
rows = []
for thr in thr_grid:
    pred_alert = (policy_df["risk_late"] >= thr).astype(int)
    alert_rate = pred_alert.mean()
    if alert_rate > max_alert_rate:
        continue

    tp = int(((pred_alert == 1) & (policy_df["is_late"] == 1)).sum())
    fp = int(((pred_alert == 1) & (policy_df["is_late"] == 0)).sum())
    fn = int(((pred_alert == 0) & (policy_df["is_late"] == 1)).sum())

    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    cost = fp * 15 + fn * 155

    rows.append({
        "threshold": thr,
        "alert_rate": alert_rate,
        "precision": prec,
        "recall": rec,
        "business_cost": cost,
    })

policy_table = pd.DataFrame(rows).sort_values("business_cost")
policy_table.head(10).round(4)

,threshold,alert_rate,precision,recall,business_cost
0,0.18,0.1778,0.1841,0.3486,168090
1,0.19,0.1299,0.1750,0.2422,182220
2,0.20,0.0738,0.1596,0.1256,196945
3,0.21,0.0673,0.1617,0.1160,197745
5,0.23,0.0313,0.1552,0.0517,204735
4,0.22,0.0316,0.1535,0.0517,204810
15,0.33,0.0023,0.1818,0.0044,209345
14,0.32,0.0023,0.1818,0.0044,209345
13,0.31,0.0023,0.1818,0.0044,209345
12,0.30,0.0023,0.1818,0.0044,209345


In [19]:
best_policy = policy_table.iloc[0].to_dict()

print("Best policy under 20% alert cap")
for k, v in best_policy.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

# Quantile-based risk bands keep segments populated for ops triage.
band_df = policy_df.copy()
band_labels = ["Low", "Medium", "High", "Critical"]
band_df["risk_band"] = pd.qcut(
    band_df["risk_late"],
    q=4,
    labels=band_labels,
    duplicates="drop",
)

band_summary = (
    band_df.groupby("risk_band", observed=False)
    .agg(
        orders=("is_late", "size"),
        late_rate=("is_late", "mean"),
        avg_pred_margin=("pred_margin", "mean"),
    )
    .reset_index()
)
band_summary["late_rate"] = (band_summary["late_rate"] * 100).round(2)
band_summary.round(3)

Best policy under 20% alert cap
threshold: 0.1800
alert_rate: 0.1778
precision: 0.1841
recall: 0.3486
business_cost: 168090.0000


,risk_band,orders,late_rate,avg_pred_margin
0,Low,3829,1.07,16.733999
1,Medium,3857,4.15,10.320000
2,High,3629,15.51,6.471000
3,Critical,3107,18.99,-0.271000
